# 弱監督與語音評估

[完整講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/e2-weak-supervision-wer.md) · [在 Colab 執行](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/e2_weak_supervision_wer.ipynb)

本練習使用自編小型資料，在 CPU 執行。先手算替換、刪除和插入，再按組比較模型與自動標籤的 WER。

弱監督可以用來源較廣、品質參差的標籤訓練模型。例如音訊旁的文字可能是字幕、摘要或自動轉錄。這類資料可增加覆蓋範圍，但要抽查標籤錯誤是否集中在特定語言或口音，再用獨立的人工參考轉錄評估。

## 一句話要改幾個詞？

字詞錯誤率（WER）先將模型輸出和參考文字逐詞對齊，再找最少要做多少次替換 $S$、刪除 $D$、插入 $I$。分母是參考文字的詞數 $N$：

$$\mathrm{WER}=\frac{S+D+I}{N}.$$

例如參考是 \`please open the door\`，模型輸出 \`please close door\`。把 \`open\` 換成 \`close\`，刪掉 \`the\`，需要 2 次編輯，WER 為 $2/4=50\%$。參考只有一詞而輸出多出兩詞時，插入數可以超過分母，因此 WER 可能高於 100%。中文等情境要先定義分詞口徑，也可以另報字元錯誤率（CER）。

## 先按組算，再解讀整體 WER

以下是自編範例。A、B 可代表不同錄音條件或不同口音組，參考文字都由教師提供。

| 群組 | 錯誤數 | 參考詞數 | WER |
|---|---:|---:|---:|
| A | 1 | 8 | 12.5% |
| B | 4 | 8 | 50% |
| 合計 | 5 | 16 | 31.25% |

整體 WER 用全部錯誤數除以全部參考詞數。這個例子給兩組相同詞數；若詞數不同，整體分數會較受長音訊群組影響。Notebook 另外算自動標籤相對人工參考的 WER，示範如何檢查訓練標籤與最終評估的品質。群組差距也可能來自麥克風、噪聲或文本難度，要先控制或分層再談原因。

## 執行環境

Colab 通常已有 NumPy 和 Matplotlib。若要使用本專案規定範圍內的套件，可執行下一格。

In [ ]:
%pip -q install 'numpy>=1.26,<3' 'matplotlib>=3.8,<4'

## 程式驗證

先預測結果，再執行下方的手算核對和視覺化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def edit_counts(reference, prediction):
    ref, pred = reference.split(), prediction.split()
    edits = [[0] * (len(pred) + 1) for _ in range(len(ref) + 1)]
    move = [[None] * (len(pred) + 1) for _ in range(len(ref) + 1)]
    for i in range(1, len(ref) + 1):
        edits[i][0], move[i][0] = i, "D"
    for j in range(1, len(pred) + 1):
        edits[0][j], move[0][j] = j, "I"
    for i, reference_word in enumerate(ref, 1):
        for j, predicted_word in enumerate(pred, 1):
            mismatch = reference_word != predicted_word
            candidates = (
                (edits[i - 1][j - 1] + mismatch, "S" if mismatch else "match"),
                (edits[i - 1][j] + 1, "D"),
                (edits[i][j - 1] + 1, "I"),
            )
            edits[i][j], move[i][j] = min(candidates, key=lambda option: option[0])

    i, j = len(ref), len(pred)
    errors = {"S": 0, "D": 0, "I": 0}
    while i or j:
        action = move[i][j]
        if action in errors:
            errors[action] += 1
        if action in ("match", "S", "D"):
            i -= 1
        if action in ("match", "S", "I"):
            j -= 1
    return errors, len(ref)

def corpus_wer(pairs):
    totals = [edit_counts(reference, output) for reference, output in pairs]
    return sum(sum(errors.values()) for errors, _ in totals) / sum(n for _, n in totals)

data = {
    "A": [
        ("open the door now", "open the door now", "open the door"),
        ("please close the door", "please close door", "please close the door"),
    ],
    "B": [
        ("open the window now", "open the door right now", "open the door"),
        ("please close the window", "close window", "close door"),
    ],
}
assert edit_counts("please open the door", "please close door")[0] == {"S": 1, "D": 1, "I": 0}
assert corpus_wer([("hello", "hello there again")]) == 2
wer_actual = [corpus_wer([(ref, hyp) for ref, hyp, _ in items]) for items in data.values()]
wer_labels = [corpus_wer([(ref, label) for ref, _, label in items]) for items in data.values()]
overall = corpus_wer([(ref, hyp) for items in data.values() for ref, hyp, _ in items])
np.testing.assert_allclose(wer_actual, [0.125, 0.5])
np.testing.assert_allclose(overall, 0.3125)
print("Model WER by group:", wer_actual, "; pooled:", overall)
print("Automatic-label WER by group:", wer_labels)

positions = np.arange(len(data))
fig, ax = plt.subplots()
ax.bar(positions - 0.18, wer_labels, width=0.35, label="Automatic labels")
ax.bar(positions + 0.18, wer_actual, width=0.35, label="Model output")
ax.set(xticks=positions, xticklabels=data, ylabel="WER", title="Audit each group separately")
ax.legend()
plt.show()

## 自我檢核與練習

把其中一組加入一筆長句，重算各組和整體 WER。解釋為什麼整體 WER 要加總錯誤與參考詞數，以及群組差距還需要哪些資料才能解釋。

## 來源

- [Whisper 原論文](https://arxiv.org/abs/2212.04356)
- [NIST 語音評估工具](https://www.nist.gov/itl/iad/mig/tools)